# Try SendSoon in Google Colab

This notebook calls the same SendSoon HTTP APIs used by the [`sendsoon/ai`](https://github.com/sendsoon/ai) MCP tools, so you can try them in the browser without installing Node.js, pnpm, or an MCP client.

| Tool | What this notebook tests |
| --- | --- |
| `ip_lookup` | Look up geolocation and ISP for a public IP |
| `markitdown_convert` | Convert a sample file to Markdown |
| `send_email` | Send one test email (optional recipient + API Key) |

Colab runtimes are temporary. Do not paste a real API Key into a shared notebook, and do not commit one to Git.

To use these tools from Cursor, Claude, or Codex, follow the [project README](https://github.com/sendsoon/ai#readme) and configure the local MCP server.

## 1. Configure

Keep the API base URL as `https://sendsoonai.com`.

- **Recipient email**: required only for `send_email`. Leave it empty to skip that cell.
- **API Key**: optional. Leave it empty to use the anonymous trial. One public IP can send up to three free test emails per day. Generate a Key at [Profile](https://sendsoonai.com/profile).

Google Colab shares outbound IPs, so the anonymous email quota may already be used. Use a Key if `send_email` returns a quota error.

In [ ]:
import os
from getpass import getpass

try:
    import requests
except ImportError:
    import subprocess
    import sys

    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "requests"])
    import requests

os.environ["SENDSOON_API_BASE_URL"] = "https://sendsoonai.com"

recipient = input("Recipient email for send_email (leave blank to skip): ").strip()
if recipient:
    os.environ["SENDSOON_EMAIL_RECIPIENT"] = recipient
else:
    os.environ.pop("SENDSOON_EMAIL_RECIPIENT", None)

api_key = getpass("API Key (optional, press Enter to skip): ").strip()
if api_key:
    os.environ["SENDSOON_API_KEY"] = api_key
else:
    os.environ.pop("SENDSOON_API_KEY", None)
del api_key

print("Base URL:", os.environ["SENDSOON_API_BASE_URL"])
print("Recipient:", os.environ.get("SENDSOON_EMAIL_RECIPIENT") or "(not set, send_email will be skipped)")
print("API Key:", "configured" if os.environ.get("SENDSOON_API_KEY") else "not set (anonymous trial)")

## 2. API helpers

These helpers wrap the same endpoints as the MCP tools: `/api/ip/lookup`, `/api/markitdown/convert`, and `/api/send-test-email`.

In [ ]:
import html
import json
import uuid
from typing import Any

def base_url() -> str:
    return os.environ["SENDSOON_API_BASE_URL"].rstrip("/")

def auth_headers() -> dict[str, str]:
    headers = {"Accept": "application/json"}
    api_key = os.environ.get("SENDSOON_API_KEY")
    if api_key:
        headers["Authorization"] = f"Bearer {api_key}"
    return headers

def show_response(response: requests.Response) -> Any:
    print(f"HTTP {response.status_code}")
    content_type = (response.headers.get("content-type") or "").lower()
    if "application/json" in content_type:
        data = response.json()
        print(json.dumps(data, indent=2, ensure_ascii=False))
        return data
    text = response.text.strip()
    print(text[:4000] if text else "(empty body)")
    return text

def ip_lookup(ip: str) -> Any:
    response = requests.get(
        f"{base_url()}/api/ip/lookup",
        params={"ip": ip},
        headers=auth_headers(),
        timeout=30,
    )
    return show_response(response)

def markitdown_convert(filename: str, content: bytes) -> Any:
    headers = auth_headers()
    headers["Accept"] = "text/markdown, application/json"
    response = requests.post(
        f"{base_url()}/api/markitdown/convert",
        headers=headers,
        files={"file": (filename, content)},
        timeout=60,
    )
    return show_response(response)

def send_email(to: str, subject: str, body: str) -> Any:
    headers = {
        **auth_headers(),
        "Content-Type": "application/json",
        "Idempotency-Key": str(uuid.uuid4()),
    }
    payload = {
        "to": to,
        "subject": subject,
        "htmlContent": (
            '<pre style="white-space:pre-wrap;font-family:inherit">'
            f"{html.escape(body)}</pre>"
        ),
    }
    response = requests.post(
        f"{base_url()}/api/send-test-email",
        headers=headers,
        json=payload,
        timeout=30,
    )
    return show_response(response)

print("Helpers ready")

## 3. Test `ip_lookup`

Look up `8.8.8.8`. This cell does not require an API Key or recipient email.

In [ ]:
ip_lookup("8.8.8.8")

## 4. Test `markitdown_convert`

Convert a small in-memory text file to Markdown. You can change `filename` and `sample` to try another format the API supports, such as `.html` or `.csv`.

In [ ]:
sample = """# SendSoon Colab sample

This file is converted through the same API as the `markitdown_convert` MCP tool.

- Look up a public IP
- Convert a file to Markdown
- Send a test email
"""

result = markitdown_convert("sendsoon-colab-sample.txt", sample.encode("utf-8"))
if isinstance(result, dict) and result.get("markdown"):
    print("\n--- Markdown preview ---\n")
    print(result["markdown"])

## 5. Test `send_email`

This cell runs only when you entered a recipient in the configuration cell. The address sent to the API is that same recipient. Without an API Key, one public IP can send up to three free test emails per day.

In [ ]:
recipient = os.environ.get("SENDSOON_EMAIL_RECIPIENT", "").strip()
if not recipient:
    print("Skipped: enter a recipient email in the configuration cell to try send_email.")
else:
    send_email(
        to=recipient,
        subject="SendSoon Colab test",
        body="Configuration successful. This message was sent from the Google Colab notebook.",
    )